[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L19.ipynb)

# Lecture 19 — Debugging PyTorch

*Read the error, isolate the line, fix it fast. The last lecture of Volume I.*

**PyTorch from Ground Up · Volume I · Chapter 34** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Read a traceback bottom-up: what, where, and the next clue.
- Find shape bugs with prints, asserts and `torchinfo.summary`.
- Fix device and dtype mismatches with one-line changes.
- Catch a `nan` or exploding loss and triage it cheapest-first.
- Find the silent bugs: missing `zero_grad`, in-place operations, a frozen loss.
- Run the same eight-step debugging workflow every time.

In [ ]:
# This lecture uses torchinfo — installed here if missing (Colab doesn't ship them)
import importlib.util, subprocess, sys
for pkg, mod in [('torchinfo', 'torchinfo')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Practitioners debug faster

Practitioners don’t write bug-free code. They **debug faster**.

<sub>Source: lecture slides 3</sub>

## 02 · Reading a traceback

Read a traceback **bottom-up**: the last line says what broke; climb to the deepest line **in your own code**. The lines in between trace the calls that led there, most recent last.

<sub>Source: lecture slides 4–7</sub>

🧪 *Interactive lab in the app:* **Read it bottom to top** — try it there to see this idea move.

## 03 · Shape bugs and the tools that find them

Shape mismatches cause **more errors than everything else combined**. `(32x784 and 256x10)`: a Linear built for **256** features received **784**. The fix is nearly always a wrong `in_features`, a missing flatten, or an earlier reshape.

<sub>Source: lecture slides 8–15</sub>

🧪 *Interactive lab in the app:* **Print shapes everywhere** — try it there to see this idea move.

**`model.py`** — predict the output, then run.

In [ ]:
def forward(self, x):
    assert x.dim() == 4, \
        f"want (B,C,H,W), got {tuple(x.shape)}"
    return self.net(x)

<details><summary>▸ Expected output</summary>

```
AssertionError: want (B,C,H,W), got (3, 32, 32)
```
</details>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(16 * 32 * 32, 10))
    def forward(self, x):
        return self.net(x)


**`summary.py`** — predict the output, then run.

In [ ]:
from torchinfo import summary

model = MyModel()
summary(model, input_size=(1, 3, 32, 32))

<details><summary>▸ Expected output</summary>

```
Layer (type:depth-idx)        Output Shape         Param #
======================================================
MyModel                       [1, 10]              --
├─Conv2d: 1-1                 [1, 16, 32, 32]      448
├─ReLU: 1-2                   [1, 16, 32, 32]      --
├─Flatten: 1-3                [1, 16384]           --
├─Linear: 1-4                 [1, 10]              163,850
======================================================
Total params: 164,298
```
</details>

## 04 · Device and dtype mismatches

Devices: **one `device`, everything through `.to(device)`**. Dtypes: **labels `long`, features `float32`**. Labels index classes, so they’re integers. NumPy hands you float64; cast on conversion.

<sub>Source: lecture slides 16–21</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(torch.randn(64, 3, 32, 32), torch.randint(0, 10, (64,))), batch_size=32)
criterion = nn.CrossEntropyLoss()


**`train.py`** — predict the output, then run.

In [ ]:
device = "cuda" if torch.cuda.is_available() \
    else "cpu"
model = model.to(device)

for x, y in loader:
    x, y = x.to(device), y.to(device)
    # ... now everything matches ...

## 05 · NaNs and an exploding loss

The bugs that **don’t throw**: the loss just prints `nan`, or rockets to `inf`. Usual suspects: a learning rate that’s too high, unnormalised inputs, a `log(0)` in a custom loss, a corrupted sample.

<sub>Source: lecture slides 22–28</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


**`detect_nan.py`** — predict the output, then run.

In [ ]:
for step, (x, y) in enumerate(loader):
    loss = criterion(model(x), y)
    if torch.isnan(loss) or torch.isinf(loss):
        print("bad loss at step", step)
        break
    loss.backward()

**`anomaly.py`** — predict the output, then run.

In [ ]:
with torch.autograd.detect_anomaly():
    loss = criterion(model(x), y)
    loss.backward()

<details><summary>▸ Expected output</summary>

```
Error detected in LogBackward0. Traceback of forward call that caused the error:
    loss = (x * torch.log(x)).sum()
RuntimeError: Function 'LogBackward0' returned nan values in its 0th output.
```
</details>

> ⚠️ **Watch out — A diagnostic, not a setting**  
> `detect_anomaly` slows training dramatically, because it checks every operation. Use it to locate a NaN, then turn it off.
>
> Never leave it on in a real training run.

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
loss = criterion(model(x), y)


**`clip.py`** — predict the output, then run.

In [ ]:
loss.backward()
torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=1.0)
optimizer.step()

## 06 · Silent bugs in autograd and the loop

The quietest family: **no error at all** — the model just won’t learn. The cause lives in autograd and the training loop.

<sub>Source: lecture slides 29–33</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
loss = criterion(model(x), y)
mask = x > 0


**`order_inplace.py`** — predict the output, then run.

In [ ]:
optimizer.zero_grad()   # clear old gradients
loss.backward()         # compute new ones
optimizer.step()        # apply them

# Risky: mutates a tensor autograd may need
x += 1
x[mask] = 0
# Safe: creates a new tensor
x = x + 1
x = x.masked_fill(mask, 0)

> ⚠️ **Watch out — A frozen loss**  
> If your loss doesn’t change at all across epochs, check three things in order: (1) did you call `optimizer.step()`? (2) are the parameters actually passed to the optimizer? (3) is anything detached from the graph with `.detach()` or a stray `no_grad()` that shouldn’t be there?
>
> A frozen loss almost always means gradients aren’t reaching the weights. (Detaching the whole output makes `backward()` raise; the silent version is rebuilding the loss from a plain number, like `torch.tensor(loss.item(), requires_grad=True)`.)

## 07 · A repeatable workflow — and the end of Volume I

When something breaks, don’t poke at random. **Run the same sequence every time.** You’ll find most bugs in minutes rather than hours.

<sub>Source: lecture slides 34–38</sub>

🧪 *Interactive lab in the app:* **The bug clinic** — try it there to see this idea move.

## The rule — worth remembering

**Errors that throw — recap**

1. Traceback: bottom line = what; deepest line of yours = where.
2. Shapes: print through `forward`, `assert` your assumptions, run `summary` first.
3. Devices: one `device`, everything through `.to(device)`.
4. Dtypes: `.long()` labels, `float32` features.

**A repeatable debugging workflow — eight rules**

1. **Read the traceback bottom-up.** Name the error, then find the deepest line in your own code.
2. **Shape errors:** print shapes through `forward`, or run `torchinfo.summary` to see the whole map at once.
3. **Device errors:** route every model and batch through one `.to(device)`.
4. **Dtype errors:** labels for cross-entropy must be `.long()`; features usually `float32`.
5. **NaN or exploding loss:** check inputs, drop the learning rate 10×, add gradient clipping, then `detect_anomaly()`.
6. **Model won’t learn:** confirm `zero_grad → backward → step`, and that nothing is silently detached.
7. **Reproduce small.** Shrink to one batch and one step. A bug you can trigger in two seconds is a bug you can fix.
8. **Change one thing at a time.** Random edits create new bugs faster than they fix old ones.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — break a model on purpose

Give an `nn.Linear(256, 10)` an input of shape (32, 784) and run it. Read the traceback: find the error type on the bottom line and your own offending line above it.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x784 and 256x10)`. The bottom line is the *what*; the line where you called the layer is the *where*. Train this muscle before you need it.
</details>

### Exercise 02 · Ex 1

Feed a (32, 784) tensor into `nn.Linear(256, 10)`. Fix it two ways: correct `in_features`, and reshape the input. Confirm both run. (Then check the output’s batch size.)

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Fix 1: `nn.Linear(784, 10)` → output (32, 10). Fix 2: `x.reshape(-1, 256)` — but 32 × 784 = 25,088 = 98 × 256, so the output is (98, 10): it *runs*, yet the batch size is no longer 32. A reshape that silences an error can still be wrong — which is why you check the output.
</details>

### Exercise 03 · Ex 2 & 3

Build a small CNN, print `x.shape` after each layer, run one batch. Where does the spatial size shrink? Then run `summary` on that CNN with `input_size=(1, 3, 32, 32)`. Do its shapes match your prints?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The spatial size shrinks at pooling layers and at convolutions without padding (or with stride > 1); a 3×3 convolution with `padding=1` keeps it. `summary` shows the same shapes with a batch of 1 in place of your batch size.
</details>

### Exercise 04 · Ex 4

Put the model on `"cuda"` (or simulate it on CPU), leave the batch on CPU. Read the error, fix it. What’s the minimal change?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`Expected all tensors to be on the same device…` The minimal change is one line at the top of the loop: `x, y = x.to(device), y.to(device)`.
</details>

### Exercise 05 · Ex 5

Train a tiny model with `lr=10` until the loss blows up; catch the step with the `isnan` check. Then lower the LR and add clipping. Does it recover?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The check fires within a few steps — and the loss may be `inf` rather than `nan`, which is why you test both. With the learning rate lowered 10× or more and `clip_grad_norm_(…, max_norm=1.0)` between `backward()` and `step()`, the loss falls normally (restart from fresh weights: `nan` parameters don’t heal).
</details>

### Exercise 06 · Ex 6

Remove `optimizer.zero_grad()` from a working loop. Watch the loss over a few epochs. Explain the pattern, then restore the line.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The loss stops decreasing, then oscillates and grows: every batch’s gradient is added to all the earlier ones, so each update is too large and points in a stale direction.
</details>

### Exercise 07 · Ex 7

Wrap a step in `detect_anomaly()` and add a `torch.log` of a tensor containing a zero. Where does it report the NaN? (If nothing is reported, print the loss, then try `x * torch.log(x)`.)

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`log(0)` alone gives `-inf`, not `nan`, so nothing may be flagged. `x * torch.log(x)` with a zero in x produces 0 × (−inf) = `nan`, and the report names `LogBackward0` and shows the forward line that created it.
</details>

### Interview-style

“A colleague’s loss is stuck at exactly the same value every epoch. List the three most likely causes in order, and the one-line check you’d run for each.” Five to six sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

A loss that never changes almost always means gradients aren’t reaching the weights. First, `optimizer.step()` may be missing or never reached — I’d print one parameter before and after a step to see whether it changes. Second, the parameters may not have been passed to the optimizer, for instance sub-modules kept in a plain Python list — I’d compare `len(list(model.parameters()))` with what I expect. Third, something may be cut out of the graph by a `.detach()`, a stray `torch.no_grad()`, or a loss rebuilt from `loss.item()` — I’d check `loss.requires_grad` and whether `p.grad` is `None` after `backward()`. I’d test all of this on a single batch and a single step, changing one thing at a time.
</details>